# Adaptive baseline mortality

Native analysis for Figure 3 and Supplementary Figure 3. The registered residual subtracts the model's weather and infection contributions. Weekly fitted baseline rates are averaged across all 50 native hybrid checkpoints.

[Original analysis notebook](https://github.com/ClimSocAna/sari-weather-mortality/blob/4402a80d3e490ad7964a3be12597043761bebda1/notebooks/03_adaptive_baseline_mortality.ipynb).

In [1]:
from importlib.util import module_from_spec, spec_from_file_location
from pathlib import Path
import sys

cwd = Path.cwd()
repo = next(
    (
        path for path in (cwd, *cwd.parents)
        if (path / "configs" / "training.example.yaml").is_file()
    ),
    None,
)
if repo is None:
    raise FileNotFoundError("Open the notebook from the sari-weather-mortality checkout.")

sys.path.insert(0, str(repo))
sys.path.insert(0, str(repo / "src"))

import matplotlib.pyplot as plt
import numpy as np
import torch

# Load helpers from this checkout even if Jupyter cached another notebooks package.
utils_path = repo / "notebooks" / "utils.py"
spec = spec_from_file_location("sari_notebook_utils", utils_path)
notebook_utils = module_from_spec(spec)
spec.loader.exec_module(notebook_utils)

ensemble = notebook_utils.ensemble
weekly_sum = notebook_utils.weekly_sum
ANALYSIS_PERIOD = notebook_utils.ANALYSIS_PERIOD
DEVICE = notebook_utils.DEVICE
checkpoint_paths = notebook_utils.checkpoint_paths

from sari_t.inference import InferenceModel
from sari_t.run.checkpoint import load_checkpoint, load_run_config, restore_predictor
from sari_t.baseline import calculate_baseline_death_rate

## Load models for inspection

Run the next cell to load member `000` for each setup into memory. Inspect `model` for the hybrid predictor or `models[setup]` for another predictor. Set `inspection_member` to another available member index to inspect its weights. The analysis cells evaluate all available members in a separate inference workspace.

In [ ]:
inspection_member = 0
setups = ['default']
checkpoints = {setup: checkpoint_paths(setup) for setup in setups}
models = {
    setup: restore_predictor(load_checkpoint(paths[inspection_member]), device="cpu").eval()
    for setup, paths in checkpoints.items()
}
model = models["default"]
model

## Check the baseline's weekly timing

The primary filter is one-sided and shifts its smoothed rates by one week after initialization. The first fitted week uses the first residual as its seed. The centered sensitivity filter uses a different timing rule.

In [ ]:
paths = checkpoints["default"]
config = load_run_config(paths[0].parents[1] / "config.yaml").adaptive_baseline_config
assert config.method == "median_ewma"
residual = torch.full((16, 15, 2), 100.0)
population = torch.full((16 * 7, 15, 2), 10_000.0)
rates = calculate_baseline_death_rate(residual, population, config)
for week in range(1, 16):
    perturbed = residual.clone()
    perturbed[week:] *= 2
    recomputed = calculate_baseline_death_rate(perturbed, population, config)
    torch.testing.assert_close(recomputed[:(week + 1) * 7], rates[:(week + 1) * 7],
                               rtol=0, atol=0)

perturbed = residual.clone()
perturbed[0] *= 2
first_week = calculate_baseline_death_rate(perturbed, population, config)[:7]
assert not torch.equal(first_week, rates[:7])
print("Weeks after initialization use completed weeks only; the first week is seeded.")

## Reconstruct the registered baseline

In [ ]:
def summarize(model, result):
    prediction = result.predictions
    additional = prediction.total_deaths - result.scenarios["reference"]
    infection = prediction.sari_deaths + prediction.covid_deaths
    observed = model.labels.weekly_de_age_sex
    residual = observed - weekly_sum((additional + infection).sum(dim=1))
    rate = weekly_sum(model.baseline_rate.squeeze(1)) / 7
    return {"Observed": observed, "Residual": residual, "Rate": rate}

paths = checkpoints["default"]
inputs = load_run_config(paths[0].parents[1] / "config.yaml").data_config
inference_model = InferenceModel.from_checkpoint(
    paths[0], data_config=inputs, period=ANALYSIS_PERIOD, device=DEVICE,
)
estimates = ensemble(inference_model, paths, summarize)

destatis = estimates["Observed"]
death_additional_weekly = destatis - estimates["Residual"]
model_rate = estimates["Rate"]

## Manuscript Figure 3: Selected age groups

Weekly registered and baseline deaths are shown above the corresponding baseline mortality rates for ages 60-64, 70-74, and 80-84. Separate left and right axes allow the male and female time trends to be compared despite different mortality levels.

The dashed series show unsmoothed residual counts. The lower panels show fitted, smoothed daily rates. The first plotted week's rate is sensitive to initialization.

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# ---- year ticks along weekly index ----
days_cumulative = 80 / 7
year_ticks = []
years = list(range(2015, 2027))

for year in years:
    year_ticks.append(days_cumulative)
    days_in_year = 366 if (year % 4 == 0 and (year % 100 != 0 or year % 400 == 0)) else 365
    days_cumulative += days_in_year / 7

age_groups = [7, 9, 11]

fig, ax = plt.subplots(2, 3, figsize=(15, 10))

for idx, age_group in enumerate(age_groups):
    # ---------- top row: weekly deaths ----------
    ax0 = ax[0, idx]

    # men – deaths (left y-axis)
    ax0.plot(
        destatis[:, age_group, 0],
        label="Registered deaths – men",
        color="C0",
        linestyle="-",
        alpha = 0.8,
    )
    ax0.plot(
        (destatis - death_additional_weekly)[:, age_group, 0],
        label="Residual deaths – men",
        color="C0",
        linestyle="--",
        alpha = 0.8,
    )
    ax0.xaxis.set_visible(False)
    # women – deaths (right y-axis)
    ax0_r = ax0.twinx()
    ax0_r.plot(
        destatis[:, age_group, 1],
        label="Registered deaths – women",
        color="C1",
        linestyle="-",
        alpha = 0.8,
    )
    ax0_r.plot(
        (destatis - death_additional_weekly)[:, age_group, 1],
        label="Residual deaths – women",
        color="C1",
        linestyle="--",
        alpha = 0.8,
    )

    ax0.text(
        0, 1.03,          # x<0 稍微往外一点，y>1 在轴上方
        "Men",
        transform=ax0.transAxes,
        ha="center",
        va="bottom",
        fontsize=12,
        fontweight='bold',
    )

    # 在右 y 轴正上方写 "Women"
    ax0_r.text(
        1, 1.03,           # x>1 往右一点，y>1 在轴上方
        "Women",
        transform=ax0_r.transAxes,
        ha="center",
        va="bottom",
        fontsize=12,
        fontweight='bold',
    )
    ax0.set_title(f"Age {25 + 5 * age_group} - {29 + 5 * age_group}", fontsize=12, y=1.05)
    ax0.grid(True, which="both", alpha=0.3)

    # ---------- bottom row: baseline mortality rates ----------
    ax1 = ax[1, idx]

    # men – baseline rate (left y-axis)
    ax1.plot(
        model_rate[:, age_group, 0],
        label="Baseline mortality rate – men",
        color="C0",
        linestyle="-",
    )

    # women – baseline rate (right y-axis)
    ax1_r = ax1.twinx()
    ax1_r.plot(
        model_rate[:, age_group, 1],
        label="Baseline mortality rate – women",
        color="C1",
        linestyle="-",
    )

    ax1.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
    ax1_r.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
    ax1.grid(True, which="both", alpha=0.3)

    # x-axis: calendar years for bottom row (shared with top)
    ax1.set_xticks(year_ticks)
    ax1.set_xticklabels(years, rotation=45, fontsize=10)

    # vertical lines at year positions on both rows
    for t in year_ticks:
        ax0.axvline(t, linestyle=":", linewidth=0.5, alpha=0.6)
        ax1.axvline(t, linestyle=":", linewidth=0.5, alpha=0.6)

# ---------- axis labels ----------
ax[0, 0].set_ylabel("Deaths per week")
ax[1, 0].set_ylabel("Baseline mortality rate")

# ---------- common legend for whole figure ----------
custom_lines = [
    Line2D([0], [0], color="C0", linestyle="-"),
    Line2D([0], [0], color="C0", linestyle="--"),
    Patch(facecolor="C0", edgecolor="none"),
    Patch(facecolor="C1", edgecolor="none"),
]

legend_labels = [
    "Registered deaths",
    "Residual deaths",
    "Men",
    "Women",
]


leg1 = fig.legend(
    custom_lines[:2],
    legend_labels[:2],
    loc="upper left",
    bbox_to_anchor=(0.01, 0.995),          # (x, y) in ax[0,0] axes coords
    bbox_transform=ax[0, 0].transAxes,
    frameon=True,
    framealpha=0.7,
    fontsize=12                     # transparency of the white box
)

# keep leg1 when creating leg2
fig.add_artist(leg1)

# second legend (colors)
leg2 = fig.legend(
    custom_lines[2:],
    legend_labels[2:],
    loc="upper left",
    bbox_to_anchor=(0.01, 0.995),
    bbox_transform=ax[0, 1].transAxes,
    frameon=True,
    framealpha=0.7,
    fontsize=12
)

plt.subplots_adjust(wspace=0.30, hspace=0.10)
# plt.savefig('death_rates.svg', bbox_inches='tight')
# plt.savefig('death_rates.png', bbox_inches='tight', dpi=300)
plt.show()

## Supplementary Figure 3: All age groups

This extended panel applies the same registered-versus-baseline comparison to all modeled age groups, including the aggregated 0-30 and open-ended 95+ categories.

In [ ]:
# ---- year ticks along weekly index ----
days_cumulative = 80 / 7
year_ticks = []
years = list(range(2015, 2026))

for year in years:
    year_ticks.append(days_cumulative)
    days_in_year = 366 if (year % 4 == 0 and (year % 100 != 0 or year % 400 == 0)) else 365
    days_cumulative += days_in_year / 7

age_groups = np.arange(0,15)

# 现在：3 行 × 2 列
fig, ax = plt.subplots(len(age_groups), 2, figsize=(18, 2 * len(age_groups)), sharex=True)

for i, age_group in enumerate(age_groups):
    # ---------- 左列：weekly deaths ----------
    ax_death = ax[i, 0]

    # women – deaths (right y-axis)
    ax_death_r = ax_death.twinx()
    ax_death_r.plot(
        destatis[:, age_group, 1],
        label="Registered deaths – women",
        color="C1",
        linestyle="-",
    )
    ax_death_r.plot(
        (destatis - death_additional_weekly)[:, age_group, 1],
        label="Residual deaths – women",
        color="C1",
        linestyle="--",
    )

    # men – deaths (left y-axis)
    ax_death.plot(
        destatis[:, age_group, 0],
        label="Registered deaths – men",
        color="C0",
        linestyle="-",
    )
    ax_death.plot(
        (destatis - death_additional_weekly)[:, age_group, 0],
        label="Residual deaths – men",
        color="C0",
        linestyle="--",
    )

    # 每一行在左图写年龄段
    if i == 0:
        ax_death.set_ylabel(f"Age 0 - 30", fontsize=12)
    elif i == 14:
        ax_death.set_ylabel(f"Age 95+", fontsize=12)
    else:
        ax_death.set_ylabel(f"Age {25 + 5 * age_group} - {29 + 5 * age_group}", fontsize=12)

    ax_death.grid(True, which="both", alpha=0.3)

    # ---------- 右列：baseline mortality rates ----------
    ax_rate = ax[i, 1]

    # men – baseline rate (left y-axis)
    ax_rate.plot(
        model_rate[:, age_group, 0],
        label="Baseline mortality rate – men",
        color="C0",
        linestyle="-",
    )

    # women – baseline rate (right y-axis)
    ax_rate_r = ax_rate.twinx()
    ax_rate_r.plot(
        model_rate[:, age_group, 1],
        label="Baseline mortality rate – women",
        color="C1",
        linestyle="-",
    )

    ax_rate.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
    ax_rate_r.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
    ax_rate.grid(True, which="both", alpha=0.3)

    # 垂直年线：两列都画
    for t in year_ticks:
        ax_death.axvline(t, linestyle=":", linewidth=0.5, alpha=0.6)
        ax_rate.axvline(t, linestyle=":", linewidth=0.5, alpha=0.6)

    # 在第一行图上方标 Men / Women
    if i == 0:
        ax_death.text(
            0.0, 1.2,
            "Men",
            transform=ax_death.transAxes,
            ha="center",
            va="bottom",
            fontsize=15,
            fontweight="bold",
        )
        ax_death_r.text(
            1.0, 1.2,
            "Women",
            transform=ax_death_r.transAxes,
            ha="center",
            va="bottom",
            fontsize=15,
            fontweight="bold",
        )
        ax_rate.text(
            0.0, 1.2,
            "Men",
            transform=ax_rate.transAxes,
            ha="center",
            va="bottom",
            fontsize=15,
            fontweight="bold",
        )
        ax_rate_r.text(
            1.0, 1.2,
            "Women",
            transform=ax_rate_r.transAxes,
            ha="center",
            va="bottom",
            fontsize=15,
            fontweight="bold",
        )

# 只在最后一行设置 x tick 和 label（共享 x）
for j in range(2):
    ax[-1, j].set_xticks(year_ticks)
    ax[-1, j].set_xticklabels(years, rotation=45, fontsize=15)

# 列标题：左=Deaths per week，右=Baseline mortality rate
ax[0, 0].set_title("Deaths per week", fontsize=15, y=1.3)
ax[0, 1].set_title("Baseline mortality rate", fontsize=15, y=1.3)

# ---------- common legend ----------
custom_lines = [
    Line2D([0], [0], color="C0", linestyle="-"),   # Registered deaths
    Line2D([0], [0], color="C0", linestyle="--"),  # Residual deaths
    Patch(facecolor="C0", edgecolor="none"),
    Patch(facecolor="C1", edgecolor="none"),
]

legend_labels = [
    "Registered deaths",
    "Residual deaths",
    "Men",
    "Women",
]

# first legend (line styles) – anchored to the first axes position
leg1 = fig.legend(
    custom_lines[:2],
    legend_labels[:2],
    loc="upper left",
    bbox_to_anchor=(0.05, 1.3),          # (x, y) in ax[0,0] axes coords
    bbox_transform=ax[0, 0].transAxes,
    ncol=2,
    frameon=True,
    framealpha=0.7,
    fontsize=12                     # transparency of the white box
)

# keep leg1 when creating leg2
fig.add_artist(leg1)

# second legend (colors)
leg2 = fig.legend(
    custom_lines[2:],
    legend_labels[2:],
    loc="upper right",
    bbox_to_anchor=(0.94, 1.3),
    bbox_transform=ax[0, 0].transAxes,
    ncol=2,
    frameon=True,
    framealpha=0.7,
    fontsize=12
)
plt.subplots_adjust(wspace=0.3, hspace=0.25)
plt.tight_layout(rect=[0, 0, 1, 0.95])
# plt.savefig('death_baseline_all.png', bbox_inches='tight', dpi=300)
plt.show()